In [1]:
%env CUDA_VISIBLE_DEVICES=7

env: CUDA_VISIBLE_DEVICES=7


In [4]:
CLOTH_BASE_KWS = [
    "fabric", "cloth", "textile", "leather", "felt", "canvas", "paper"
]

In [6]:
keywords = ['cloth']
text = 'tablecloth'
any(k in text for k in keywords)

True

In [2]:
from pathlib import Path
import tensorflow_datasets as tfds

builder_dir = "/data/gaoya/dataset/kubric_tfds/movi-d/256x256/1.0.0"
builder = tfds.builder_from_directory(builder_dir)

print(builder.info)

ds = builder.as_dataset(split="train")
sample = next(iter(tfds.as_numpy(ds.take(1))))

print(sample.keys())
print("video shape:", sample["video"].shape)
print("video dtype:", sample["video"].dtype)

I0000 00:00:1774410520.241534    6687 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1774410520.300622    6687 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/data/gaoya/miniconda3/envs/wan/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
I0000 00:00:1774410521.723948    6687 port.cc:153] oneDNN custom operations are on. You may see slightly different n

tfds.core.DatasetInfo(
    name='movi_d',
    full_name='movi_d/256x256/1.0.0',
    description="""
    A simple rigid-body simulation with GSO objects and an HDRI background.
    The scene consists of a dome (half-sphere) onto which a random HDRI is projected, 
    which acts as background, floor and lighting.
    The scene contains between 10 and 20 random static objects, and between 1 and 3
    dynamic objects (tossed onto the others).
    The camera position is sampled randomly in a half-sphere shell around the scene
    and always points at the origin.
    
    Static objects are spawned without overlap in the region [(-7, -7, 0), (7, 7, 10)],
    and are simulated to fall and settle before the first frame of the scene.
    Dynamic objects are spawned without overlap in the region [(-5, -5, 1), (5, 5, 5)], and
    initialized with a random velocity from the range [(-4, -4, 0), (4, 4, 0)]
    minus the position of the object to bias their trajectory towards the center of
    the sc

I0000 00:00:1774410541.332091    6687 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46590 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:d8:00.0, compute capability: 8.9
I0000 00:00:1774410542.486267    7420 tf_record_dataset_op.cc:396] The default buffer size is 262144, which is overridden by the user specified `buffer_size` of 8388608


dict_keys(['background', 'backward_flow', 'camera', 'depth', 'events', 'forward_flow', 'instances', 'metadata', 'normal', 'object_coordinates', 'segmentations', 'video'])
video shape: (24, 256, 256, 3)
video dtype: uint8


In [ ]:
'''{
  "metadata": {
    "video_name": int,                    # 视频/样本编号
    "depth_range": (2,),                 # depth 反量化范围 [min, max]
    "forward_flow_range": (2,),          # 前向光流反量化范围 [min, max]
    "backward_flow_range": (2,),         # 后向光流反量化范围 [min, max]
    "num_frames": 24,                    # 总帧数
    "num_instances": int,                # 场景中的物体实例数
    "height": 256,                       # 图像高度
    "width": 256                         # 图像宽度
  },

  "background": str,                     # 背景 HDRI 名称

  "camera": {
    "field_of_view": 0.85755605,         # 相机视场角（弧度）
    "focal_length": 35.0,                # 焦距
    "positions": (24, 3),                # 24 帧相机位置，每帧一个 (x, y, z)
    "quaternions": (24, 4),              # 24 帧相机姿态四元数
    "sensor_width": 32.0                 # 传感器宽度
  },

  "instances": {
    # 以下字段都是“按物体实例”组织的；
    # 第一维 nr_instances 表示场景中的物体个数

    "angular_velocities": (nr_instances, 24, 3),   # 每个物体 24 帧角速度，(wx, wy, wz)

    "asset_id": (nr_instances,),                   # 每个物体在 Google Scanned Objects (GSO) 数据集中的资产 ID

    "bbox_frames": TensorShape([nr_instances, None]), # 每个物体在哪些帧有 2D bbox；None 表示每个物体可见帧数不固定

    "bboxes": TensorShape([nr_instances, None, 4]),   # 每个物体在可见帧上的 2D bbox，通常是 (xmin, ymin, xmax, ymax)

    "bboxes_3d": (nr_instances, 24, 8, 3),         # 每个物体每帧的 3D 包围盒 8 个角点

    "category": (nr_instances,),                   # 每个物体的类别，取值属于固定集合：
                                                   # ["Action Figures", "Bag", "Board Games",
                                                   #  "Bottles and Cans and Cups", "Camera",
                                                   #  "Car Seat", "Consumer Goods", "Hat",
                                                   #  "Headphones", "Keyboard", "Legos",
                                                   #  "Media Cases", "Mouse", "None",
                                                   #  "Shoe", "Stuffed Toys", "Toys"]

    "friction": (nr_instances,),                   # 每个物体的摩擦系数

    "image_positions": (nr_instances, 24, 2),      # 每个物体每帧投影到图像中的中心/参考点 2D 坐标

    "is_dynamic": (nr_instances,),                 # 每个物体是否为动态物体：
                                                   # False = 场景开始时已静止放在地面
                                                   # True  = 场景开始后被抛入场景

    "mass": (nr_instances,),                       # 每个物体质量

    "positions": (nr_instances, 24, 3),            # 每个物体每帧 3D 位置

    "quaternions": (nr_instances, 24, 4),          # 每个物体每帧姿态四元数

    "restitution": (nr_instances,),                # 每个物体恢复系数/弹性系数

    "scale": (nr_instances,),                      # 每个物体的缩放系数，范围通常在 0.75 ~ 3.0

    "velocities": (nr_instances, 24, 3),           # 每个物体每帧线速度，(vx, vy, vz)

    "visibility": (nr_instances, 24)               # 每个物体每帧可见性，可理解为可见像素数或可见程度
  },

  "events": {
    "collisions": {
      "contact_normal": (2778, 3),      # 每次碰撞的接触法向量
      "force": (2778,),                 # 每次碰撞的作用力大小
      "frame": (2778,),                 # 每次碰撞发生的帧号
      "image_position": (2778, 2),      # 每次碰撞点投影到图像平面的 2D 坐标
      "instances": (2778, 2),           # 每次碰撞涉及的两个物体实例索引
      "position": (2778, 3)             # 每次碰撞发生的 3D 位置
    }
  },

  "depth": (24, 256, 256, 1),              # 深度图：24 帧，每帧单通道
  "forward_flow": (24, 256, 256, 2),       # 前向光流：每像素 2 维位移
  "backward_flow": (24, 256, 256, 2),      # 后向光流：每像素 2 维位移
  "normal": (24, 256, 256, 3),             # 法线图：每像素 3 维法向量
  "object_coordinates": (24, 256, 256, 3), # 物体坐标图：每像素对应的物体局部/归一化坐标
  "segmentations": (24, 256, 256, 1),      # 实例分割图：每像素是实例 ID
  "video": (24, 256, 256, 3)               # RGB 视频帧
}'''

In [4]:
from pathlib import Path
import tensorflow_datasets as tfds
import imageio.v2 as imageio

builder_dir = "/data/gaoya/dataset/kubric_tfds/movi-d/256x256/1.0.0"
builder = tfds.builder_from_directory(builder_dir)
ds = builder.as_dataset(split="train")

sample = next(iter(tfds.as_numpy(ds.take(1))))
video = sample["video"]   # shape 大概率是 [24, 256, 256, 3]

out_path = Path("/home/gaoya/Code_Video/Code_data/vis/movi_d_sample0.mp4")
writer = imageio.get_writer(out_path, fps=12)

for frame in video:
    writer.append_data(frame)

writer.close()
print(f"saved: {out_path}")
print("video shape:", video.shape)

saved: /home/gaoya/Code_Video/Code_data/vis/movi_d_sample0.mp4
video shape: (24, 256, 256, 3)
